<a href="https://colab.research.google.com/github/naubergois/curso-transfer-colab/blob/main/notebooks/01-o-peso-que-viajou.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 1. O peso que viajou

Uma rede nova nasce sem opinião. Os pesos são o que ela já viu. Transfer learning pega esses pesos, treinados em milhões de fotos do ImageNet, e leva para um caso com poucas fotos suas.

A rede usada aqui é a ResNet-18. A última camada, a cabeça, responde 1000 classes: cachorro, carro, cadeira. A base, tudo que vem antes, descreve a foto num vetor. Troca a cabeça. A base fica.

> No Colab: **Ambiente de execução → Executar tudo**. CPU basta. O arquivo de pesos baixa uma vez.

In [ ]:
!pip install -q torch torchvision

## A base chega pronta, a cabeça sai

`resnet18(weights=...)` baixa o ofício. `rede` fica intacta, para a gente contar o que veio. `modelo` é a cópia de trabalho: a base inteira fica com `requires_grad = False`, e a cabeça vira uma camada linear de duas saídas.

Duas saídas são o caso novo. Círculo e quadrado, ou as duas classes que forem suas. Isso ainda não treina nada. Só mostra o que viajou e o que ficou livre.

In [ ]:
def contar(modelo, so_treinavel=False):
    total = 0
    for p in modelo.parameters():
        if so_treinavel and not p.requires_grad:
            continue
        total += p.numel()
    return total


import torch
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(7)

rede = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
rede.eval()
entrada = rede.fc.in_features
saida_antiga = rede.fc.out_features

modelo = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
modelo.eval()
for p in modelo.parameters():
    p.requires_grad = False
modelo.fc = nn.Linear(entrada, 2)

print(f"pesos que vieram: {contar(rede)}")
print(f"cabeça antiga: {entrada} -> {saida_antiga}")
print(f"cabeça nova: {modelo.fc.in_features} -> {modelo.fc.out_features}")
print(f"ainda treinam: {contar(modelo, so_treinavel=True)}")
print(f"parados: {contar(modelo) - contar(modelo, so_treinavel=True)}")
print(f"conv1 pode mudar? {modelo.conv1.weight.requires_grad}")

## Exercícios

Seis contas em cima do que já está na memória. Escreva a resposta na variável `resposta` e rode a célula: `conferir` diz na hora se fechou. Se travar, abra **Ver solução**.

In [ ]:
# @title Preparação dos exercícios (rode, não precisa ler)
VERIFICAR = {
    1: lambda r: int(r) == contar(rede),
    2: lambda r: int(r) == saida_antiga,
    3: lambda r: int(r) == entrada,
    4: lambda r: int(r) == contar(modelo, so_treinavel=True),
    5: lambda r: int(r) == contar(modelo) - contar(modelo, so_treinavel=True),
    6: lambda r: bool(r) is False,
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

### Exercício 1. Quantos pesos vieram

Quantos parâmetros a `rede` pré-treinada tem, antes de trocar a cabeça?

<details>
<summary>Ver solução</summary>

```python
resposta = contar(rede)
```

</details>

In [ ]:
# Dica: contar(rede) soma o numel de cada parâmetro.
resposta = None

conferir(1, resposta)

### Exercício 2. A cabeça antiga

Quantas classes a cabeça antiga sabia responder? É o `out_features` da `rede`.

<details>
<summary>Ver solução</summary>

```python
resposta = rede.fc.out_features
```

</details>

In [ ]:
# Dica: rede.fc.out_features
resposta = None

conferir(2, resposta)

### Exercício 3. O tamanho do vetor

Quantos números a base entrega para a cabeça? É o `in_features`.

<details>
<summary>Ver solução</summary>

```python
resposta = rede.fc.in_features
```

</details>

In [ ]:
# Dica: A variável entrada já guardou esse número. rede.fc.in_features também.
resposta = None

conferir(3, resposta)

### Exercício 4. O que ainda treina

Depois de congelar a base e trocar a cabeça, quantos parâmetros de `modelo` ainda podem mudar?

<details>
<summary>Ver solução</summary>

```python
resposta = contar(modelo, so_treinavel=True)
```

</details>

In [ ]:
# Dica: contar(modelo, so_treinavel=True) ignora quem está com requires_grad False.
resposta = None

conferir(4, resposta)

### Exercício 5. O que ficou parado

Quantos parâmetros de `modelo` ficaram congelados?

<details>
<summary>Ver solução</summary>

```python
resposta = contar(modelo) - contar(modelo, so_treinavel=True)
```

</details>

In [ ]:
# Dica: O total menos os que ainda treinam.
resposta = None

conferir(5, resposta)

### Exercício 6. A primeira camada

O peso `modelo.conv1.weight` ainda pode mudar? Responda `True` ou `False`.

<details>
<summary>Ver solução</summary>

```python
resposta = modelo.conv1.weight.requires_grad
```

</details>

In [ ]:
# Dica: requires_grad diz se o otimizador tem permissão de mexer.
resposta = None

conferir(6, resposta)

In [ ]:
placar()

## O que fica

Mil classes emprestadas não são o seu problema. O número que importa é o dos pesos que ainda podem mudar: só a cabeça nova. O viés do ImageNet viaja junto com a base. Gente, cão e carro estavam naquele saco. O seu caso pode não estar.

Os seis exercícios abaixo usam estes mesmos nomes: `rede`, `modelo`, `entrada`, `saida_antiga` e `contar`.